<a href="https://colab.research.google.com/github/Omar-Anwar-Dev/news-category-classification/blob/main/notebooks/00_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# News Category Classification — Main Notebook

Multi-class news classifier (42 categories) using headline + short description.

**Pipeline:** Kaggle dataset → preprocessing → TF-IDF + numeric features → 6 classical models + RoBERTa-embedding SVM → Gradio UI with Groq LLM explanations and similarity retrieval.

Per ADR-009, all pipeline code lives in this notebook. The only Python file maintained outside is `src/preprocessing.py` (because its `clean_text()` is unit-tested in CI).

Section headers map directly to sprint-1 tasks; later sprints fill in the remaining models and the full Gradio UI.

## Colab Quick-Start (run-once cell below)

If you opened this notebook on Colab, run the cell below first. It clones
the repo, installs dependencies, downloads NLTK corpora, and pulls Groq +
Kaggle credentials from the **Colab Secrets** panel (key icon on the left
sidebar) so the rest of the notebook runs unchanged.

If you opened this on your laptop with a local venv, the cell is a no-op —
your existing setup keeps working.

**Required Colab Secrets** (sidebar → key icon → "+ Add new secret",
toggle "Notebook access" on for each):
- `GROQ_API_KEY` — from https://console.groq.com/keys
- `KAGGLE_USERNAME` and `KAGGLE_KEY` — open `kaggle.json` in any text editor; copy `"username"` and `"key"` values

In [ ]:
"""Colab one-time bootstrap — safe to re-run; safe on local Windows/Linux."""

import json
import os
import sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    REPO_DIR = "/content/news-category-classification"
    REPO_URL = "https://github.com/Omar-Anwar-Dev/news-category-classification.git"

    if not os.path.exists(REPO_DIR):
        print(f"Cloning {REPO_URL} ...")
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)

    print("Installing dependencies (first run: ~3 min)...")
    os.system("pip install -q -r requirements.txt")

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    import nltk

    for r in ["punkt", "punkt_tab", "stopwords", "wordnet"]:
        nltk.download(r, quiet=True)

    # --- Groq key from Colab Secrets ---
    try:
        from google.colab import userdata

        if not os.environ.get("GROQ_API_KEY"):
            try:
                os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
                print("OK: GROQ_API_KEY loaded from Colab Secrets")
            except Exception:
                print("WARN: GROQ_API_KEY not found in Colab Secrets — Groq cell will skip.")
    except ImportError:
        pass

    # --- Kaggle credentials from Colab Secrets (write kaggle.json) ---
    kaggle_json = os.path.join(REPO_DIR, "kaggle.json")
    if not os.path.exists(kaggle_json):
        try:
            from google.colab import userdata

            creds = {
                "username": userdata.get("KAGGLE_USERNAME"),
                "key": userdata.get("KAGGLE_KEY"),
            }
            with open(kaggle_json, "w") as f:
                json.dump(creds, f)
            os.chmod(kaggle_json, 0o600)
            print("OK: kaggle.json written from Colab Secrets")
        except Exception:
            print(
                "WARN: Kaggle credentials not in Colab Secrets. Either add "
                "KAGGLE_USERNAME and KAGGLE_KEY to Secrets, or upload kaggle.json "
                "manually via the Files panel on the left."
            )

    print("\nColab bootstrap complete.")
else:
    print("Running locally — bootstrap skipped (assumes venv + .env are configured).")

## Setup & Imports

Runs once at the top of the notebook. Idempotent — safe to re-run.

In [ ]:
import logging
import os
import sys
from pathlib import Path

# Make `src/` importable when running from the notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load .env if present (local Windows / Linux). On Colab use Colab Secrets UI instead.
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    pass

# Standard project paths.
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EMBEDDINGS_DIR = DATA_DIR / "embeddings"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFUSION_DIR = REPORTS_DIR / "confusion"
for d in (RAW_DIR, PROCESSED_DIR, EMBEDDINGS_DIR, MODELS_DIR, REPORTS_DIR, CONFUSION_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Single source of truth for randomness (ADR-section: reproducibility).
RANDOM_STATE = 42

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("main")
log.info(f"Project root: {PROJECT_ROOT}")

## 1. Data Loading — S1-T5

Download the News Category Dataset (~209K rows) from Kaggle. The first run hits Kaggle; subsequent runs read from local cache under `data/raw/`.

In [ ]:
"""S1-T5 — Kaggle download with local cache + JSON-Lines parse.

Returns a DataFrame with columns: link, headline, category,
short_description, authors, date. About 209K rows (full file ~80 MB).
First call hits Kaggle; subsequent calls read from `data/raw/`.
"""

import json

import pandas as pd

DATASET_SLUG = "rmisra/news-category-dataset"
DATASET_FILENAME = "News_Category_Dataset_v3.json"


def load_dataset(force_download: bool = False) -> pd.DataFrame:
    cache_path = RAW_DIR / DATASET_FILENAME
    if cache_path.exists() and not force_download:
        log.info(f"Using cached dataset at {cache_path} ({cache_path.stat().st_size / 1e6:.1f} MB)")
    else:
        # KaggleApi reads credentials from KAGGLE_CONFIG_DIR/kaggle.json.
        os.environ["KAGGLE_CONFIG_DIR"] = str(PROJECT_ROOT)
        from kaggle.api.kaggle_api_extended import KaggleApi

        api = KaggleApi()
        api.authenticate()
        log.info(f"Downloading {DATASET_SLUG} from Kaggle...")
        api.dataset_download_files(DATASET_SLUG, path=str(RAW_DIR), unzip=True)
        log.info(f"Cached to {cache_path}")

    rows = []
    with cache_path.open("r", encoding="utf-8") as fh:
        for line in fh:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    log.info(f"Loaded {len(df):,} articles | columns: {list(df.columns)}")
    return df


raw_df = load_dataset()
raw_df.head()

## 2. Preprocessing — S1-T6

Calls `clean_text()` from `src/preprocessing.py` (kept as a module so it can be unit-tested in CI). Adds three numeric features (word / char / punct counts) and writes the cleaned dataset to `data/processed/cleaned.parquet`.

In [ ]:
"""S1-T6 — Apply clean_text + numeric features over the dataset.

Reads `raw_df` from the cell above, produces the cleaned dataset with
the cleaned text column plus three numeric features
(word_count, char_count, punct_count), and persists to
`data/processed/cleaned.parquet`. Cached on subsequent runs.
"""

from tqdm.auto import tqdm

from src.preprocessing import build_numeric_features, clean_text

CLEANED_PATH = PROCESSED_DIR / "cleaned.parquet"

if CLEANED_PATH.exists():
    log.info(f"Using cached cleaned dataset at {CLEANED_PATH}")
    cleaned_df = pd.read_parquet(CLEANED_PATH)
else:
    log.info(f"Cleaning {len(raw_df):,} articles (~3-5 min on a laptop CPU)...")
    combined = (raw_df["headline"].fillna("") + " " + raw_df["short_description"].fillna("")).tolist()

    feats = [build_numeric_features(t) for t in tqdm(combined, desc="numeric")]
    texts = [clean_text(t) for t in tqdm(combined, desc="clean")]

    cleaned_df = pd.DataFrame(
        {
            "text": texts,
            "category": raw_df["category"].values,
            "word_count": [f["word_count"] for f in feats],
            "char_count": [f["char_count"] for f in feats],
            "punct_count": [f["punct_count"] for f in feats],
        }
    )
    cleaned_df.to_parquet(CLEANED_PATH, compression="snappy")
    log.info(f"Saved cleaned dataset to {CLEANED_PATH}")

log.info(f"Cleaned dataset: {len(cleaned_df):,} rows x {cleaned_df.shape[1]} columns")
cleaned_df.head()

## 2.5. Category Consolidation — S2-T1

Per **ADR-010**, the raw 42 labels in the dataset include several near-duplicates (`STYLE` vs `STYLE & BEAUTY`, the three `ARTS` variants, etc.). Sprint 1's confusion matrix shows these are the dominant source of error. Consolidating them to **27 final classes** raises classical models from ~55% to ~66% accuracy and fine-tuned RoBERTa to ~70%.

This cell applies the merge map, drops rows whose cleaned text is shorter than 4 words, drops exact duplicates, and overwrites `data/processed/cleaned.parquet` with the 27-class dataset. All sprint-2 models train against this consolidated label space.

In [ ]:
"""S2-T1 — Category consolidation 42 → 27, drop short / duplicate rows.

Re-saves `data/processed/cleaned.parquet` in place. All downstream cells
(features, classical training, RoBERTa eval, EDA part 2, Gradio) operate
on this 27-class dataset.
"""

CATEGORY_MAP = {
    "STYLE & BEAUTY": "STYLE",
    "THE WORLDPOST": "POLITICS_WORLD",
    "WORLDPOST": "POLITICS_WORLD",
    "POLITICS": "POLITICS_WORLD",
    "WELLNESS": "HEALTH",
    "HEALTHY LIVING": "HEALTH",
    "TASTE": "FOOD",
    "FOOD & DRINK": "FOOD",
    "ARTS": "ARTS",
    "CULTURE & ARTS": "ARTS",
    "ARTS & CULTURE": "ARTS",
    "COLLEGE": "EDUCATION",
    "EDUCATION": "EDUCATION",
    "SCIENCE": "SCI_TECH",
    "TECH": "SCI_TECH",
    "GREEN": "ENVIRONMENT",
    "ENVIRONMENT": "ENVIRONMENT",
    "MONEY": "BUSINESS",
    "BUSINESS": "BUSINESS",
    "WOMEN": "FAMILY",
    "PARENTS": "FAMILY",
    "PARENTING": "FAMILY",
    "CRIME": "CRIME",
    "WEIRD NEWS": "CRIME",
    "MEDIA": "ENTERTAINMENT",
    "ENTERTAINMENT": "ENTERTAINMENT",
    "STYLE": "STYLE",
}
KEEP_AS_IS = {
    "SPORTS", "TRAVEL", "RELIGION", "COMEDY", "QUEER VOICES",
    "BLACK VOICES", "DIVORCE", "FIFTY", "GOOD NEWS", "HOME & LIVING",
    "IMPACT", "LATINO VOICES", "U.S. NEWS", "WEDDINGS", "WORLD NEWS",
}
MIN_SAMPLES_PER_CLASS = 1000
MIN_WORDS = 4


def _consolidate(cat: str) -> str | None:
    if cat in CATEGORY_MAP:
        return CATEGORY_MAP[cat]
    if cat in KEEP_AS_IS:
        return cat
    return None  # drop unmapped


log.info(f"Before consolidation: {cleaned_df['category'].nunique()} categories, {len(cleaned_df):,} rows")

# 1. Map labels.
cleaned_df = cleaned_df.assign(category=cleaned_df["category"].map(_consolidate)).dropna(subset=["category"])

# 2. Drop short cleaned texts.
n_before = len(cleaned_df)
cleaned_df = cleaned_df[cleaned_df["text"].str.split().str.len() >= MIN_WORDS]
log.info(f"Dropped {n_before - len(cleaned_df):,} rows with <{MIN_WORDS}-word cleaned text")

# 3. Drop exact duplicates.
n_before = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates(subset=["text", "category"])
log.info(f"Dropped {n_before - len(cleaned_df):,} exact duplicates")

# 4. Drop classes below the minimum sample count (defensive).
counts = cleaned_df["category"].value_counts()
small = counts[counts < MIN_SAMPLES_PER_CLASS].index.tolist()
if small:
    cleaned_df = cleaned_df[~cleaned_df["category"].isin(small)]
    log.warning(f"Dropped {len(small)} classes below {MIN_SAMPLES_PER_CLASS}: {small}")

cleaned_df = cleaned_df.reset_index(drop=True)
cleaned_df.to_parquet(CLEANED_PATH, compression="snappy")
log.info(
    f"After consolidation: {cleaned_df['category'].nunique()} categories, "
    f"{len(cleaned_df):,} rows. Saved to {CLEANED_PATH}"
)
cleaned_df["category"].value_counts()

## 2.6. Download Fine-tuned RoBERTa — S2-T2

Per **ADR-011** (supersedes ADR-003), the project's primary classifier is a `RobertaForSequenceClassification` model fine-tuned on the 27-class merged dataset, sourced from the team coordinator's prior individual work (see `docs/decisions.md` for full attribution).

The model package lives on Google Drive (**ADR-012**) at FILE_ID `19EIWqmmR4tbJrMyiqKYRT__s_d1n11rW` and weighs ~470 MB. First call downloads + extracts it; subsequent calls reuse the local cache under `models/best_model/`.

In [ ]:
"""S2-T2 — Download (or reuse cached) fine-tuned RoBERTa, then load it.

Network call only on the first run. After extraction, `models/best_model/`
contains config.json, model.safetensors, tokenizer.json,
tokenizer_config.json, training_args.bin.
"""

import subprocess
import zipfile

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DRIVE_FILE_ID = "19EIWqmmR4tbJrMyiqKYRT__s_d1n11rW"  # ADR-012
BEST_MODEL_DIR = MODELS_DIR / "best_model"
BEST_MODEL_ZIP = MODELS_DIR / "best_model.zip"

if (BEST_MODEL_DIR / "config.json").exists():
    log.info(f"Using cached fine-tuned model at {BEST_MODEL_DIR}")
else:
    log.info(f"Downloading best_model.zip from Drive (~470 MB) ...")
    try:
        import gdown
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown

    gdown.download(id=DRIVE_FILE_ID, output=str(BEST_MODEL_ZIP), quiet=False)

    log.info(f"Extracting {BEST_MODEL_ZIP.name} ...")
    with zipfile.ZipFile(BEST_MODEL_ZIP) as zf:
        zf.extractall(MODELS_DIR)
    BEST_MODEL_ZIP.unlink()
    log.info(f"Extracted to {BEST_MODEL_DIR}")

# Load model + tokenizer.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
roberta_model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_DIR).to(device).eval()
log.info(
    f"Loaded {roberta_model.__class__.__name__} | "
    f"num_labels={roberta_model.config.num_labels} | device={device}"
)
assert roberta_model.config.num_labels == 27, (
    f"Expected 27 labels, got {roberta_model.config.num_labels}"
)

# Sanity check on 5 random samples.
sample = cleaned_df.sample(5, random_state=RANDOM_STATE)[["text", "category"]].reset_index(drop=True)
with torch.no_grad():
    enc = tokenizer(
        sample["text"].tolist(),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    ).to(device)
    logits = roberta_model(**enc).logits
    pred_ids = logits.argmax(dim=-1).cpu().numpy()
predicted = [roberta_model.config.id2label[int(i)] for i in pred_ids]
print("Sanity check (5 random samples):")
for (_, row), pred in zip(sample.iterrows(), predicted):
    mark = "✓" if pred == row.category else " "
    print(f"  {mark} true=[{row.category:>15}]  pred=[{pred:>15}]  | {row.text[:60]}")

## 3. EDA

Exploratory analysis lives in `notebooks/01_eda.ipynb`. Sprint 1 produces part 1 (shape / missing / class distribution); sprint 2 adds the full chart set.

## 4. Classical Features — S1-T8

TF-IDF (uni+bi-grams, capped at 50K features, sublinear TF) stacked with three numeric features (word / char / punct counts, scaled). Persisted to `models/`.

In [ ]:
"""S1-T8 — TF-IDF (uni+bi-gram, 50K cap, sublinear, min_df=2) + numeric features.

Stratified 80/20 split with `random_state=RANDOM_STATE`, vectorizer fit on
TRAIN ONLY (PRD F4), scaler fit on TRAIN ONLY. Persists vectorizer,
numeric scaler and label encoder to `models/`. Includes the round-trip
test (fit → save → load → transform produces an identical matrix).

Outputs in scope: X_train, X_test, y_train, y_test, plus the raw text
arrays for downstream display. Subsequent cells (S1-T9 train, S1-T10
evaluate) reuse these.
"""

import joblib
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Encode labels -> integer ids; persist for downstream inference.
label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(cleaned_df["category"].values)
joblib.dump(label_encoder, MODELS_DIR / "label_encoder.joblib")
log.info(f"Label encoder fit on {len(label_encoder.classes_)} classes; saved.")

# Stratified 80/20 split — same indices reused for every model in sprint 2.
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    cleaned_df["text"].values,
    cleaned_df[["word_count", "char_count", "punct_count"]].to_numpy(),
    y_all,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_all,
)
log.info(f"Train: {len(X_text_train):,}   Test: {len(X_text_test):,}")

# Fit TF-IDF on TRAIN only, transform both.
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,
    min_df=2,
)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)
joblib.dump(tfidf, MODELS_DIR / "tfidf_vectorizer.joblib")
log.info(f"TF-IDF vocab: {len(tfidf.vocabulary_):,} features")

# Fit StandardScaler on TRAIN numeric only, transform both.
scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)
joblib.dump(scaler, MODELS_DIR / "numeric_scaler.joblib")

# Sparse-stack TF-IDF + scaled numeric (3 cols).
X_train = hstack([X_tfidf_train, csr_matrix(X_num_train_scaled)]).tocsr()
X_test = hstack([X_tfidf_test, csr_matrix(X_num_test_scaled)]).tocsr()
log.info(f"X_train: {X_train.shape}   X_test: {X_test.shape}   nnz/row: {X_train.nnz / X_train.shape[0]:.1f}")

# Round-trip test (S1-T8 acceptance).
tfidf_reload = joblib.load(MODELS_DIR / "tfidf_vectorizer.joblib")
scaler_reload = joblib.load(MODELS_DIR / "numeric_scaler.joblib")
sample_idx = slice(0, 5)
roundtrip = hstack(
    [tfidf_reload.transform(X_text_test[sample_idx]), csr_matrix(scaler_reload.transform(X_num_test[sample_idx]))]
).tocsr()
diff = X_test[sample_idx] - roundtrip
assert diff.nnz == 0, f"Round-trip mismatch: {diff.nnz} non-zero deltas"
log.info("Round-trip test passed: load(saved).transform == original.transform on 5 sample rows")

## 5. Train Logistic Regression Baseline — S1-T9

Stratified 80/20 split, `class_weight='balanced'`, `GridSearchCV` over `C ∈ {0.1, 1, 10}`. Persists best estimator to `models/logreg_best.joblib`.

In [ ]:
"""S1-T9 — LogReg baseline with GridSearchCV.

Logistic Regression with `class_weight='balanced'`, multinomial via the
`saga` solver (good for large sparse text features), grid search over
`C ∈ {0.1, 1, 10}` with 3-fold CV scoring on macro-F1.

`saga` chosen over `lbfgs` for memory/speed on sparse 50K-dim input;
over `liblinear` because we want multinomial probabilities, not OvR.
Persists the best refit-on-full-train estimator to
`models/logreg_best.joblib`.
"""

import time

import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV

base = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    max_iter=300,
    random_state=RANDOM_STATE,
    n_jobs=1,  # let GridSearchCV parallelise across folds/params
)

search = GridSearchCV(
    base,
    param_grid={"C": [0.1, 1.0, 10.0]},
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)

t0 = time.time()
log.info("Starting GridSearchCV(cv=3) over C in {0.1, 1.0, 10.0} ...")
search.fit(X_train, y_train)
log.info(f"Done in {(time.time() - t0) / 60:.1f} min")
log.info(f"Best C: {search.best_params_['C']}, mean CV macro-F1: {search.best_score_:.4f}")

logreg_best = search.best_estimator_
joblib.dump(logreg_best, MODELS_DIR / "logreg_best.joblib")

# Quick test-set sanity check (full evaluation suite runs in S1-T10).
y_pred = logreg_best.predict(X_test)
test_macro_f1 = f1_score(y_test, y_pred, average="macro")
test_weighted_f1 = f1_score(y_test, y_pred, average="weighted")
log.info(f"Test set: macro-F1 = {test_macro_f1:.4f} | weighted-F1 = {test_weighted_f1:.4f}")
assert test_macro_f1 >= 0.45, f"S1-T9 acceptance fail: macro-F1 {test_macro_f1:.4f} < 0.45"
log.info("S1-T9 acceptance met (macro-F1 >= 0.45).")

## 6. Evaluate Models — S1-T10

Computes (accuracy, precision, recall, F1-macro, F1-weighted, ROC-AUC OvR) for each persisted model and appends a row to `reports/model_comparison.csv`. Saves a confusion-matrix PNG to `reports/confusion/<name>.png`.

In [ ]:
"""S1-T10 — `evaluate_model()` skeleton + first row in comparison table.

Computes (accuracy, precision_macro, recall_macro, f1_macro, f1_weighted,
roc_auc_ovr_macro) for any fitted sklearn classifier. Appends a row to
`reports/model_comparison.csv` (replacing any prior row for the same model)
and saves a row-normalised confusion-matrix PNG to
`reports/confusion/<name>.png`.

Sprint 2's models call this same function — that's why the per-model
row-replacement logic exists.
"""

from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

COMPARISON_CSV = REPORTS_DIR / "model_comparison.csv"


def evaluate_model(model, name, X_test, y_test, label_encoder=None):
    """Compute metrics + persist comparison row + confusion-matrix PNG."""
    y_pred = model.predict(X_test)

    metrics = {
        "model": name,
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
        metrics["roc_auc_ovr"] = roc_auc_score(y_test, proba, multi_class="ovr", average="macro")
    else:
        metrics["roc_auc_ovr"] = float("nan")

    # Append/replace row in comparison CSV.
    new_row = pd.DataFrame([metrics])
    if COMPARISON_CSV.exists():
        existing = pd.read_csv(COMPARISON_CSV)
        existing = existing[existing["model"] != name]
        out = pd.concat([existing, new_row], ignore_index=True)
    else:
        out = new_row
    out.to_csv(COMPARISON_CSV, index=False)
    log.info(f"Wrote '{name}' row to {COMPARISON_CSV.name} ({len(out)} model rows total)")

    # Confusion matrix.
    classes = label_encoder.classes_ if label_encoder is not None else np.unique(y_test)
    cm = confusion_matrix(y_test, y_pred, normalize="true")
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        cm,
        ax=ax,
        cmap="Blues",
        vmin=0,
        vmax=1,
        xticklabels=classes,
        yticklabels=classes,
        square=True,
        cbar_kws={"shrink": 0.7},
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion matrix (row-normalised) — {name}")
    plt.xticks(rotation=90, fontsize=7)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    cm_path = CONFUSION_DIR / f"{name}.png"
    fig.savefig(cm_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    log.info(f"Saved confusion matrix to {cm_path}")

    return metrics


# Run on the LogReg baseline.
metrics = evaluate_model(logreg_best, "logreg", X_test, y_test, label_encoder)
print(f"LogReg baseline:")
for k, v in metrics.items():
    fmt = f"{v:.4f}" if isinstance(v, float) else str(v)
    print(f"  {k:<18}: {fmt}")

## 7. RoBERTa Feasibility Spike — S1-T11

Embed 100 sample sentences through `roberta-base`, mean-pool, save `(100, 768)` to `data/embeddings/roberta_spike.npy`. Logs peak memory and wall-time so sprint 2 can size the full extraction. **Run on Colab GPU** — local CPU is too slow for the full version.

In [ ]:
"""S1-T11 — RoBERTa feasibility spike.

Embed 100 sample sentences through `roberta-base`, mean-pool the last
hidden states (excluding padding tokens), save `(100, 768)` float32 to
`data/embeddings/roberta_spike.npy`. Logs peak GPU memory and wall-time
so sprint 2 can size the full extraction in S2-T6.

**Run on Colab GPU runtime.** Even on a free-tier T4 this should be a
few seconds for 100 sentences — the timing tells us how much headroom
the full 209K-row pass will have. Local Windows CPU also works (~30s
for 100 samples) but is not the target environment for the full-scale
embedding pass.

The mean-pooling + caching pattern here is the prototype reused
verbatim by S2-T6 over the full dataset.
"""

import time

import numpy as np
import torch
from transformers import RobertaModel, RobertaTokenizerFast

ROBERTA_SPIKE_NPY = EMBEDDINGS_DIR / "roberta_spike.npy"

if ROBERTA_SPIKE_NPY.exists():
    log.info(f"Using cached spike at {ROBERTA_SPIKE_NPY}")
    emb = np.load(ROBERTA_SPIKE_NPY)
else:
    sample_texts = cleaned_df["text"].sample(100, random_state=RANDOM_STATE).tolist()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"Device: {device}")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
    model = RobertaModel.from_pretrained("roberta-base").to(device).eval()

    t0 = time.time()
    pooled_chunks = []
    BATCH = 16
    with torch.no_grad():
        for i in range(0, len(sample_texts), BATCH):
            batch = sample_texts[i : i + BATCH]
            enc = tokenizer(
                batch, padding=True, truncation=True, max_length=128, return_tensors="pt"
            ).to(device)
            out = model(**enc).last_hidden_state  # (B, T, 768)
            mask = enc["attention_mask"].unsqueeze(-1).float()  # (B, T, 1)
            pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            pooled_chunks.append(pooled.cpu().numpy())

    emb = np.concatenate(pooled_chunks, axis=0).astype(np.float32)
    np.save(ROBERTA_SPIKE_NPY, emb)

    elapsed = time.time() - t0
    log.info(
        f"Embedded {len(emb)} sentences -> shape {emb.shape}, dtype {emb.dtype} "
        f"in {elapsed:.1f}s ({len(emb) / elapsed:.0f} sent/s)"
    )
    if device.type == "cuda":
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
        full_min = (len(cleaned_df) / len(emb)) * elapsed / 60
        log.info(f"Peak GPU memory: {peak_mb:.0f} MB")
        log.info(f"Extrapolated wall-time for full {len(cleaned_df):,}-row pass: ~{full_min:.0f} min")
    else:
        log.info("CPU run — GPU memory not measured. Move to Colab for sprint-2 full extraction.")

log.info(f"Spike embeddings: shape={emb.shape}, dtype={emb.dtype}")

## 8. Groq LLM Smoke Test — S1-T13

Auths against Groq with `GROQ_API_KEY`, sends a one-line prompt, prints the response. Verifies the API path before the full LLM integration in sprint 2.

In [ ]:
"""S1-T13 — Groq client smoke test.

Authenticates against Groq using `GROQ_API_KEY` (loaded from `.env` by
the setup cell on local Windows; on Colab put it in the Secrets UI and
load via `userdata.get('GROQ_API_KEY')`). Sends a one-line prompt,
prints the response. Verifies the API path before the full LLM
integration in S2-T10. Failure paths (missing key, network error,
rate limit) print a clear error message — they do **not** raise.
"""

GROQ_MODEL = "llama-3.3-70b-versatile"  # ADR-001
GROQ_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_KEY:
    log.error(
        "GROQ_API_KEY is not set. Add it to .env at project root "
        "(see .env.example) or to the Colab Secrets UI. Smoke test skipped."
    )
else:
    try:
        from groq import Groq

        client = Groq(api_key=GROQ_KEY)
        completion = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": "Reply with exactly the single word: pong"}],
            max_tokens=20,
            temperature=0.0,
        )
        reply = completion.choices[0].message.content
        log.info(
            f"Groq OK | model={GROQ_MODEL} | reply={reply!r} | "
            f"tokens={completion.usage.total_tokens} | latency={completion.usage.total_time:.3f}s"
        )
        print(f"Response from {GROQ_MODEL}: {reply}")
    except Exception as e:
        log.error(f"Groq smoke test FAILED: {type(e).__name__}: {e}")
        print("Groq smoke test failed — see log above. The notebook continues running.")

## 9. Minimal Gradio App — S1-T12

Two text inputs (headline, short description), one Predict button. Displays the LogReg baseline's predicted category + confidence. Launches a public share link from Colab. The full UI (LLM explanation + similar articles) ships in sprint 2.

In [ ]:
"""S1-T12 — Minimal Gradio app.

Two text inputs (headline + description), one Predict button. Returns
the LogReg baseline's predicted category and confidence (%).

Sprint 2 (S2-T12) extends this UI with: best-of-7 model selection,
Groq LLM explanation, and top-3 similar-article retrieval. The sprint-1
version exists to prove the full path from raw text to a category
prediction shown in the UI — the M0 demo deliverable.

Launches a Gradio public share link via `share=True`. Runs from Colab
(public URL valid ~72 h) or locally on http://127.0.0.1:7860.
"""

import gradio as gr
from scipy.sparse import csr_matrix, hstack

from src.preprocessing import build_numeric_features, clean_text


def _featurize_one(headline: str, description: str):
    """Reproduce the train-time feature pipeline for a single inference example."""
    raw = (headline or "").strip() + " " + (description or "").strip()
    cleaned = clean_text(raw)
    tfidf_vec = tfidf.transform([cleaned])
    nums = build_numeric_features(raw)
    num_vec = scaler.transform([[nums["word_count"], nums["char_count"], nums["punct_count"]]])
    return hstack([tfidf_vec, csr_matrix(num_vec)]).tocsr()


def predict_news(headline: str, description: str):
    """Gradio backend: text in -> (category, confidence_pct)."""
    if not (headline or "").strip() and not (description or "").strip():
        return "Please enter a headline or description.", 0.0
    X = _featurize_one(headline, description)
    probs = logreg_best.predict_proba(X)[0]
    top_idx = int(probs.argmax())
    label = label_encoder.inverse_transform([top_idx])[0]
    confidence_pct = round(float(probs[top_idx]) * 100, 1)
    return label, confidence_pct


demo = gr.Interface(
    fn=predict_news,
    inputs=[
        gr.Textbox(label="Headline", placeholder="e.g. Apple unveils new iPhone with improved camera"),
        gr.Textbox(label="Short description (optional)", lines=3),
    ],
    outputs=[
        gr.Textbox(label="Predicted category"),
        gr.Number(label="Confidence (%)"),
    ],
    title="News Category Classification — Sprint 1 (M0)",
    description="Logistic-Regression baseline over TF-IDF + numeric features. Sprint 2 adds the LLM explanation and similar-article retrieval.",
    allow_flagging="never",
    examples=[
        ["Apple unveils new iPhone with improved camera and battery life", "The latest model emphasises low-light photography and a 25% larger battery."],
        ["Senate passes sweeping climate bill in late-night vote", "The bill commits $370B to clean energy and electric vehicle subsidies over the next decade."],
        ["Lakers beat Celtics in overtime thriller", "LeBron James scored 38 points as the Lakers held off a fourth-quarter Celtics rally."],
    ],
)

# In Colab, `share=True` produces a public link valid ~72h.
demo.launch(share=True)